In [1]:
%pip install beautifulsoup4 playwright pandas matplotlib seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 596.5 kB/s  0:01:10m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 595.6 kB/s  0:00:16eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 598.1 kB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 584.6 kB/s  0:00:04 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 591.3 kB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 618.0 kB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [seaborn]4/15 [seaborn]ib]

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
from bs4 import BeautifulSoup
import re
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

TARGET_URL = "https://gateway.iisertvm.ac.in:4443/"

In [3]:
async def get_dashboard_html(username, password):
    """
    Logs into the user portal asynchronously and returns the HTML of the 
    authenticated dashboard page. Returns None if login fails or errors out.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(ignore_https_errors=True)
        page = await context.new_page()
        
        page.set_default_timeout(30000)

        try:
            await page.goto(TARGET_URL, wait_until="domcontentloaded")
            
            # Wait 2 seconds for the portal's JavaScript (UserPortalLogin.js) to initialize
            await asyncio.sleep(2)
            
            await page.fill("input#username", username)
            await page.fill("input#password", password)
            
            # Directly evaluate the login function to bypass UI overlay/rendering issues
            await page.evaluate("callLogin()")
            
            # Wait for backend session creation and the subsequent redirect
            await asyncio.sleep(4)
            
            message_div = page.locator("div#message")
            error_text = ""
            if await message_div.count() > 0:
                error_text = (await message_div.inner_text()).strip()
            
            # If we are still on mode=1 (login page) and there's an error, login failed
            if "mode=1" in page.url and error_text:
                return None
            
            html_content = await page.content()
            
            # Optional: Save a copy to disk
            # with open("dashboard_preview.html", "w", encoding="utf-8") as f:
            #     f.write(html_content)
            #     print("Written to file: dashboard_preview.html")
                
            # print(f"Process Complete for '{username[:-2]}'")
            
            return html_content

        except (PlaywrightTimeoutError, Exception):
            # Quietly return None on any connection or timeout errors
            return None
            
        finally:
            await browser.close()

In [4]:
def extract_traffic_rows(html_content):
    # Parse the raw HTML string
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # These are the exact, unchanging IDs of the labels inside the <tr> blocks we want
    target_labels = [
        "Language.UploadedData",
        "Language.DownloadedData",
        "Language.DataTrasfer"
    ]
    
    extracted_html_blocks = []
    
    for label_id in target_labels:
        # Find the label element
        label_elem = soup.find('label', id=label_id)
        
        if label_elem:
            # Find the closest <tr> tag that wraps this label
            parent_tr = label_elem.find_parent('tr')
            
            if parent_tr:
                # Convert the BeautifulSoup object back to a raw HTML string
                extracted_html_blocks.append(str(parent_tr))
                
    # Join the extracted rows with newlines for clean formatting
    clean_output = "\n".join(extracted_html_blocks)
    
    return clean_output

In [5]:
def parse_traffic_totals(html_rows_content):
    soup = BeautifulSoup(html_rows_content, 'html.parser')
    
    # Target label IDs for the three rows
    target_labels = [
        "Language.UploadedData",
        "Language.DownloadedData",
        "Language.DataTrasfer"
    ]
    
    extracted_totals = []
    
    for label_id in target_labels:
        # Find the label element inside the row
        label_elem = soup.find('label', id=label_id)
        
        if label_elem:
            # Locate the parent <tr> tag wrapping this label
            parent_tr = label_elem.find_parent('tr')
            
            if parent_tr:
                # Find all <td> cells within the row
                tds = parent_tr.find_all('td')
                
                # Index 4 corresponds to the right-most 'Total' column before 'Remaining'
                if len(tds) >= 5:
                    cell_text = tds[4].get_text(strip=True)
                    
                    match = re.search(r'([\d\.]+)', cell_text)
                    if match:
                        extracted_totals.append(float(match.group(1)))
                    else:
                        extracted_totals.append(0.0)
                else:
                    extracted_totals.append(0.0)
            else:
                extracted_totals.append(0.0)
        else:
            extracted_totals.append(0.0)
            
    return extracted_totals

In [6]:
async def snitch_internet_usage(username, password):
    raw_html = await get_dashboard_html(username, password)

    stripped_str = extract_traffic_rows(raw_html)
    values = parse_traffic_totals(stripped_str)

    return values

In [ ]:
creds = pd.read_csv("data/credentials.csv", index_col=False)

target_column = "Username" 

cleaned_names = creds[target_column].str[:-2].str.capitalize()
creds.insert(0, "Name", cleaned_names)

creds["Upload"] = 0.0
creds["Download"] = 0.0
creds["Total traffic"] = 0.0

creds.head()

,Name,Username,Password,Upload,Download,Total traffic
0,Aanandee,aanandee23,aanims2300323,0.0,0.0,0.0
1,Abdul,abdul23,abdims2300423,0.0,0.0,0.0
2,Abdulla,abdulla23,amaims2302923,0.0,0.0,0.0
3,Abhija,abhija23,abhims2300523,0.0,0.0,0.0
4,Abhijith,abhijith23,abhims2300623,0.0,0.0,0.0


In [8]:
for index, row in creds.iterrows():

    name = row["Name"] 
    user = row["Username"]
    passwd = row["Password"]
    
    # Await the async function to get the values
    ul, dl, tl = await snitch_internet_usage(user, passwd)

    print(f"{name} - ul: {ul}, dl: {dl}, tl: {tl}")
    
    # Optional: update your dataframe values here too!
    creds.at[index, "Upload"] = ul
    creds.at[index, "Download"] = dl
    creds.at[index, "Total traffic"] = tl

Aanandee - ul: 254678.06, dl: 1472685.94, tl: 1727364.0
Abdul - ul: 91913.34, dl: 1344664.94, tl: 1436578.28
Abdulla - ul: 146976.9, dl: 1381131.68, tl: 1528108.58
Abhija - ul: 41432.18, dl: 292613.4, tl: 334045.59
Abhijith - ul: 0.0, dl: 0.0, tl: 0.0
Abhisu - ul: 285994.87, dl: 2012464.12, tl: 2298458.99
Able - ul: 244299.25, dl: 1286295.57, tl: 1530594.83
Achuth - ul: 0.0, dl: 0.0, tl: 0.0
Adrija - ul: 74030.77, dl: 746215.02, tl: 820245.79
Adrita - ul: 75135.54, dl: 737435.94, tl: 812571.47
Agnivesh - ul: 48822.4, dl: 952914.98, tl: 1001737.38
Ajin - ul: 185180.18, dl: 2817320.0, tl: 3002500.18
Ajit - ul: 0.0, dl: 0.0, tl: 0.0
Akhil - ul: 72598.75, dl: 1833000.85, tl: 1905599.6
Akshat - ul: 289126.45, dl: 2674769.0, tl: 2963895.45
Aleef - ul: 64488.0, dl: 1171741.78, tl: 1236229.78
Amanvijay - ul: 59496.16, dl: 740100.07, tl: 799596.23
Amisha - ul: 143843.4, dl: 1433969.52, tl: 1577812.92
Amlin - ul: 55949.92, dl: 460124.09, tl: 516074.01
Anagharanjith - ul: 78340.18, dl: 749599.05,

In [9]:
creds.head()

,Name,Username,Password,Upload,Download,Total traffic
0,Aanandee,aanandee23,aanims2300323,254678.06,1472685.94,1727364.00
1,Abdul,abdul23,abdims2300423,91913.34,1344664.94,1436578.28
2,Abdulla,abdulla23,amaims2302923,146976.90,1381131.68,1528108.58
3,Abhija,abhija23,abhims2300523,41432.18,292613.40,334045.59
4,Abhijith,abhijith23,abhims2300623,0.00,0.00,0.00


In [18]:
creds.to_csv("scrapped_data.csv", index=False)